In [1]:
import time
import hashlib
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
from PIL import Image

from settings.config import RetryConfig
from services.diffusion_service import generate_image
from core.models import StableDiffusionInput
from checks.image_checks import (
    check_background_content,
    check_text_match,
    check_zone_clutter,
    check_clip_score,
)
from settings.config import SDXL_ID, SEED

PROMPTS = [
    "I Tested AI for 30 Days",
    "Why Most Startups Fail",
    "Build a SaaS in a Weekend",
    "The Dark Side of Productivity",
    "Can You Learn Coding Fast?",
    "This Morning Routine Changed Everything",
    "Inside a $1M App Idea",
    "How I Beat Procrastination",
    "AI vs Human Creativity",
    "Master Python in 10 Minutes",
    "The Truth About Passive Income",
    "I Tried No Social Media",
    "Make Money with Open Source?",
    "Design Thumbnails That Click",
    "The Science of Deep Work",
    "Build Your Own GPT App",
    "Why Your UI Looks Bad",
    "Secrets of High CTR Videos",
    "Startup Ideas That Work",
    "Fix Your Sleep in 7 Days",
    "From Zero to ML Engineer",
    "How Billionaires Think",
    "The Minimalist Workspace Setup",
    "Avoid These Coding Mistakes",
    "What Makes Content Go Viral?",
    "AI Automation for Developers",
    "Is Remote Work Dying?",
    "Learn System Design Fast",
    "Quiz: Guess the Country by Emoji",
    "Only 1% Can Solve This Riddle",
]



def build_prompt(title: str) -> tuple[str, str]:
    prompt = f"Professional youtube thumbnail with exact title '{title.upper()}'"
    negative_prompt = "humans, faces, hands, cluttering, logos, pixelated, blurry, low quality"
    return prompt, negative_prompt


def calculate_edge_density(image: Image.Image) -> dict:
    """
    Returns clutter metrics for the LLM-selected text zone.
    """

    # --- Convert to OpenCV ---
    img = np.array(image)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # --- Edge Density ---
    edges = cv2.Canny(gray, 100, 200)
    edge_density = np.sum(edges > 0) / edges.size

    return {"edge_density": float(edge_density)}


def generate_thumbnail(title: str, output_dir: Path) -> dict:
    prompt, negative_prompt = build_prompt(title)

    start = time.time()
    bg: Image.Image = generate_image(
        model_id=SDXL_ID,
        diffusion_input=StableDiffusionInput(prompt=prompt, negative_prompt=negative_prompt),
        seed=SEED,
    )
    duration = round(time.time() - start, 2)

    image_path = output_dir / f"{hashlib.md5(title.encode()).hexdigest()}.png"
    bg.save(image_path)

    artifact_result = check_background_content(bg)
    clutter_result = calculate_edge_density(bg)
    clip = check_clip_score(bg, prompt, cfg=RetryConfig())
    levenshtein_distance = check_text_match(title, bg)

    return {
        "Title": title,
        "Prompt": prompt,
        "Image Path": str(image_path),
        "Duration (s)": duration,
        "Edge Density": round(clutter_result["edge_density"], 4),
        "Artifact Probability": round(artifact_result[0]["neg_score"], 4),
        "CLIP Score": round(clip, 4),
        "Levenshtein Distance": levenshtein_distance,
        "Error": None,
    }


def run_batch(titles: list[str], output_dir: str = "outputs") -> pd.DataFrame:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    rows = []
    for title in titles:
        print(f"Generating: {title}")
        try:
            row = generate_thumbnail(title, out)
        except Exception as e:
            row = {"Title": title, "Prompt": build_prompt(title)[0], "Error": str(e)}
        rows.append(row)

    return pd.DataFrame(rows)


if __name__ == "__main__":
    df = run_batch(PROMPTS, output_dir="artifacts/runs/baseline")

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_path = f"artifacts/runs/baseline/batch_report_{timestamp}.xlsx"
    df.to_excel(report_path, index=False)
    print(f"Report saved: {report_path}")

d:\YT Thumbnail Generator\yt-thumbnail-generator\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 643.79it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 643.53it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: o

Generating: I Tested AI for 30 Days
Generating: Why Most Startups Fail
Generating: Build a SaaS in a Weekend
Generating: The Dark Side of Productivity
Generating: Can You Learn Coding Fast?
Generating: This Morning Routine Changed Everything
Generating: Inside a $1M App Idea
Generating: How I Beat Procrastination
Generating: AI vs Human Creativity
Generating: Master Python in 10 Minutes
Generating: The Truth About Passive Income
Generating: I Tried No Social Media
Generating: Make Money with Open Source?
Generating: Design Thumbnails That Click
Generating: The Science of Deep Work
Generating: Build Your Own GPT App
Generating: Why Your UI Looks Bad
Generating: Secrets of High CTR Videos
Generating: Startup Ideas That Work
Generating: Fix Your Sleep in 7 Days
Generating: From Zero to ML Engineer
Generating: How Billionaires Think
Generating: The Minimalist Workspace Setup
Generating: Avoid These Coding Mistakes
Generating: What Makes Content Go Viral?
Generating: AI Automation for Devel